# Analyses_NLP

In [ ]:
# Librairies pré-requises
# %pip install git+https://github.com/segment-any-text/wtpsplit.git
# %pip install einops
# %pip install -U spaCy
# %pip install -U bertopic pandas scikit-learn datasets plotly kaleido stopwordsiso nbformat ipykernel
# %pip install umap-learn
# !python -m spacy download fr_core_news_sm
# !python -m spacy download fr_dep_news_trf

In [1]:
import pandas as pd
import spacy
from bertopic import BERTopic
import umap
from hdbscan import HDBSCAN

from bertopic.vectorizers import ClassTfidfTransformer
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer


In [68]:
df = pd.read_csv(
    "../data/interim/df_repu_regroup.csv", low_memory=False, dtype={"ID_orateur": str}
)

In [69]:
import datetime
import locale

# Active la locale française (nécessaire pour le format)
locale.setlocale(locale.LC_TIME, "fr_FR.UTF-8")

'fr_FR.UTF-8'

***Possible de faire Étapes 1 et 2 sur AT et ensuite fusionner pour faire analyse***

## Étape 1 : Établir les paramètres du Topic Modeling 

In [4]:
docs = df["Texte_clean"]
language = "french"
language_short = language[:2]

### Définition des stops words 

In [ ]:
### Option simple ### 

# nltk.download("stopwords") # 1. Télécharge les stopwords français de NLTK
# french_stopwords = list(set(stopwords.words("french"))) # 2. Construit un liste de stopwords uniques
# vectorizer_model = CountVectorizer(stop_words=french_stopwords) 3. Crée un CountVectorizer qui supprimera les mots vides


In [5]:
### Option 2 Avec spacy ### 
nlp = spacy.load("fr_core_news_sm")  # !python -m spacy download fr_core_news_sm
french_stopwords = list(nlp.Defaults.stop_words)
vectorizer_model = CountVectorizer(stop_words=french_stopwords)

### Choisir le modèle d'embedding

The primary factor to tune is the embedding model, because it drastically impact the results of the topic model. To check if the embedding makes sense, you can plot the 2D map after dimension reduction with UMAP (`n_components=2`). Then, by exploring the map, you can assess if the embedding space created placed similar documents together or not.

At this point you can also try different values for `n_neighbors` and `n_components`. However, be aware that the influence of UMAP parameters on the final topic model is difficult to appreciate at first glance.

In [6]:
embedding_model = SentenceTransformer(
    # "all-MiniLM-L6-v2",  # fonctionne ok sur df_phrase et agrégats
    "Lajavaness/sentence-camembert-large", trust_remote_code=True # très bonne qualité !
    # "Intfloat/multilingual-e5-small" # mauvaise qualité pour les différents corpus
    #"paraphrase-multilingual-mpnet-base-v2"
    # "Alibaba-NLP/gte-multilingual-base", trust_remote_code=True  # 8128 tokens --> gros df mais via AT. 
    # "jinaai/jina-embeddings-v3", # 8192 tokens with RoPE. https://huggingface.co/jinaai/jina-embeddings-v3
    # "sentence-transformers/LaBSE"  
    #"Lajavaness/sentence-flaubert-base"   
)

print(
    "device used :", embedding_model.device
)  # Vérifie si le modèle est sur GPU ou CPU

device used : mps:0


In [ ]:
# Modèles à charger
    # "flaubert-large-cased", # https://huggingface.co/flaubert/flaubert_large_cased
    #"camembert/camembert-large", # https://huggingface.co/almanach/camembert-large

# "jinaai/jina-embeddings-v3", # 8192 tokens with RoPE. https://huggingface.co/jinaai/jina-embeddings-v3

### Choisir la granularité du modèle

Once you've chosen an embedding model, you can change the `n_neighbors` and `min_cluster_size`. Both work jointly: ***the lower these paramters, the smaller grain and more specific the topics***. 

To change these parameters, one must explicitly declare `UMAP` and `HDBSCAN` objects and pass them on to the `BERTopic` model:

In [7]:
# create an HDBSCAN and UMAP models
hdbscan_model = HDBSCAN(
    min_cluster_size=20,
    # autres paramètres par défaut
    prediction_data=True
)
umap_model = umap.UMAP(
    n_neighbors=20,
    metric="cosine",
    n_components=5,
    min_dist=0.0,
    low_memory=False
)

In [ ]:
# ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [ ]:
# topic_model = bertopic.BERTopic(language="french", vectorizer_model=vectorizer_model) 4. Instancie un modèle BERTopic pour analyser les thèmes de textes français
# # topics, probs = topic_model.fit_transform(df["Texte_clean"]) #5. Entraîne le modèle sur le corpus

## Étape 2 : Créer le modèle

In [8]:
# créer le modèle
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,  # remove stopwords after embbedings
    language = language,
    umap_model= umap_model,
    hdbscan_model=hdbscan_model,
    #ctfidf_model=ctfidf_model # (or) reduce the impact of frequent word
)

# emb = model.encode(list_of_sentences, batch_size=64, show_progress_bar=True)

In [9]:
# Fiter le modèle
topics, probs = topic_model.fit_transform(documents=docs)  # .tolist() ? pas obligatoire ?

## Étape 3 : Raffiner les Topic

In [10]:
hierarchical_topics = topic_model.hierarchical_topics(docs)

100%|██████████| 83/83 [00:00<00:00, 398.59it/s]


In [11]:
table_topic = topic_model.get_topic_info()
table_topic[:40]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3426,-1_qu_république_loi_été,"[qu, république, loi, été, bien, faire, monsie...",[Mon intervention se situe dans le cadre de la...
1,0,588,0_laïcité_culte_république_1905,"[laïcité, culte, république, 1905, loi, religi...","[Qu'il me soit permis, en préambule, d'adresse..."
2,1,412,1_école_élèves_éducation_enseignants,"[école, élèves, éducation, enseignants, scolai...","[Monsieur le ministre, en mai 2017, quelques j..."
3,2,342,2_république_républicain_ve_républicains,"[république, républicain, ve, républicains, vi...","[Et la République ?, « La République, c'est mo..."
4,3,325,3_sanitaire_crise_virus_urgence,"[sanitaire, crise, virus, urgence, santé, vacc...","[La France fait face, en outre-mer, à la pire ..."
5,4,320,4_49_censure_motion_avez,"[49, censure, motion, avez, majorité, gouverne...","[Aujourd'hui, nous avons failli. Nous avons fa..."
6,5,297,5_milliards_euros_impôt_sociale,"[milliards, euros, impôt, sociale, retraite, r...","[Monsieur le ministre, pas de surprise avec ce..."
7,6,292,6_asile_immigration_étrangers_intégration,"[asile, immigration, étrangers, intégration, r...",[Je sais que des avancées sont possibles et qu...
8,7,231,7_mer_territoires_ultramarins_continuité,"[mer, territoires, ultramarins, continuité, he...",[Nous connaissons bien le problème des prix de...
9,8,230,8_associations_contrat_association_engagement,"[associations, contrat, association, engagemen...",[Contrairement à ce qu'a dit mon collègue Alex...


In [12]:
fig_hierarchical = topic_model.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics
)
fig_hierarchical

In [16]:
topics_to_merge = [58, 26, 51, 75, 23, 1, 37, 80], [17, 34, 57, 15, 32, 7, 40, 65, 46, 29, 18, 55, 83, 48, 63, 78, 41, 50, 36, 67], [62, 3, 28], [6, 11], [5, 25], [59, 69, 22, 30, 9, 53, 44, 76, 72], [2, 79, 81, 42, 73], [56, 52, 60, 54], [71, 47, 20, 21, 82, 74], [31, 8, 33, 0, 43, 61], [68, 16, 24, 39, 70, 10, 19, 13], [77, 12, 49, 35, 27]
topic_model.merge_topics(docs, topics_to_merge)

In [19]:
topics_to_merge = [16, 6], [17, 12], [1, 15], [6, 11], [14, 9]
topic_model.merge_topics(docs, topics_to_merge)

In [20]:
table_topic = topic_model.get_topic_info()
table_topic[:40]

,Topic,Count,Name,Representation,Representative_Docs
0,-1,3426,-1_qu_loi_république_été,"[qu, loi, république, été, bien, faire, minist...",[Mon intervention se situe dans le cadre de la...
1,0,1434,0_territoires_qu_loi_état,"[territoires, qu, loi, état, république, mer, ...","[Monsieur le président, monsieur le Premier mi..."
2,1,1121,1_associations_qu_république_loi,"[associations, qu, république, loi, laïcité, e...",[Les principes républicains n'ont d'existence ...
3,2,812,2_école_élèves_éducation_enfants,"[école, élèves, éducation, enfants, scolaire, ...",[Le texte que nous allons examiner comporte vi...
4,3,783,3_police_qu_justice_sécurité,"[police, qu, justice, sécurité, loi, été, état...",[C'est un texte important que nous présentons ...
5,4,575,4_qu_république_élus_président,"[qu, république, élus, président, été, assembl...","[Monsieur le président, monsieur le Premier mi..."
6,5,490,5_santé_sanitaire_qu_crise,"[santé, sanitaire, qu, crise, été, faire, soin...",[…pour être à la hauteur de nos responsabilité...
7,6,476,6_drapeau_république_europe_européen,"[drapeau, république, europe, européen, extrêm...","[Madame la présidente, madame la ministre, mes..."
8,7,456,7_qu_euros_milliards_fiscale,"[qu, euros, milliards, fiscale, impôt, sociale...",[En combinant une assiette réduite et un taux ...
9,8,450,8_asile_mayotte_immigration_droit,"[asile, mayotte, immigration, droit, france, q...",[Je sais que des avancées sont possibles et qu...


In [21]:
fig_hierarchical = topic_model.visualize_hierarchy(
    hierarchical_topics=hierarchical_topics
)
fig_hierarchical

In [ ]:
# # Les principales fonctions à tester pour avoir un aperçu simple :

# topic_model.get_topic_info()
# topic_model.visualize_barchart()
# topic_model.visualize_topics()
# topic_model.visualize_hierarchy()
# topic_model.visualize_documents(df["texte"].to_list())

In [23]:
topic_model.visualize_barchart(
    n_words = 10, # Select the number of words to display per topic
    # topics = [0,1,2,3,4], # Select specific topics to display
    # top_n_topics = 6, # Select the first n topics to display
    # height = 300, # Adjust the height of the plot
    # width = 800 # Adjust the width of the plot 
)

In [24]:
# Optional: visualize
fig_topic_distance_map = topic_model.visualize_topics()
fig_topic_distance_map

#### 1/ Réattribuer le "bruit" à d'autres topics

In [ ]:
topics_reduced = topic_model.reduce_outliers(
    docs, topics, probs, 
    strategy="embeddings",
    threshold=0.2         # distance max acceptée
)


In [ ]:
topic_model.reduce_outliers(docs, topics, probabilities=None, strategy="c-tf-idf")

In [ ]:
topics_reduced = topic_model.reduce_outliers(
    documents = docs, 
    topics = topics, 
    probs = probs, 
    embedding_model = embedding_model,
    strategy="embeddings" 
)

### Réduire le nombre de topic

To aggregate topics, the algorithm proposed is to use the topic embedding (the mean of the document's embedding inside a cluster), compute the cosine similarity matrix and use the agglomerative clustering algorithm described [here](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html) to aggregate the topics. Once performed, all the documents are moved into a new group and keywords are re-generated.

With more than 100 topics, it is difficult to have a general idea of the main groups in the corpus. Looking back to the @fig-hierarchical-plot[^24] we can identify seven large branches we could reduce our topic model to.

In [ ]:
topic_model.reduce_topics(df["Texte_clean"], nr_topics=7 + 1) #Add one to account for the noise
#Retrieve the updated topics and probabilities
topics_reduced, probabilities_reduced = topic_model.topics_, topic_model.probabilities_

In [ ]:
# Absolute fail 
topic_info_reduced = topic_model.get_topic_info()
topic_info_reduced

In [ ]:
# Pour Explore the merging process

for iRow_reduced, topic_id in enumerate(topic_info_reduced["Topic"]):
    print(topic_info_reduced.loc[iRow_reduced, "Name"].replace("_", " "))
    og_topics_merged_to_new_topic = list(set([
        int(og_topic) 
        for og_topic, new_topic in zip(topics, topics_reduced) 
        if new_topic == topic_id
    ]))
    for og_topic in og_topics_merged_to_new_topic:
        print(
            "\t - ",
            topic_info.loc[
                topic_info["Topic"] == og_topic,
                "Name"
            ]
            .item()
            .replace("_", " ")
        ) 
    print("---")

In [ ]:
# Pour Explore the reason why a given document was clustered in a specific group

# Select a document
text_id = 3000
is_my_document = [i == text_id for i in range(len(docs))]
print(f"Doc n°{text_id}:\n{docs[text_id]}")

topics_per_class = topic_model.topics_per_class(docs, classes = is_my_document)
topics_per_class = topics_per_class.loc[topics_per_class["Class"], :].set_index("Topic")
# Retrieve the Topic Representation for comparison
topics_name = (topic_model.get_topic_info().set_index("Topic")["Name"])
topics_per_class.loc[:,"Topic Name"] = topics_name
print(topics_per_class.reset_index().to_markdown())

In [ ]:
# TODO: revoir regroupement des topics
# TODO: revoir Topic distribution
# TODO: aviser genAI sur le nom des topics ?

In [70]:
topic_labels = {
    0: "Territoires",
    1: "Laïcité",
    2: "Éducation",
    3: "Répressif",
    4: "Représentation",
    5: "Santé",
    6: "Symbolique",
    7: "Budget-social",
    8: "Intégration",
    9: "Politique étrangère",
    10: "Droits",
    11: "Vie parlementaire",
    12: "Mémoire",
    13: "Antisémitisme"
}


In [71]:
topic_model.set_topic_labels(topic_labels)

In [72]:
topics_per_class["Topic_name"] = (
    topics_per_class["Topic"].map(topic_labels)
)


In [94]:
topic_model.save("TM_Test_Regroup_Camembert_20")

2025-12-18 20:19:22,407 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


## Étape 4 : Analyses synchroniques

In [ ]:
from bertopic import BERTopic
topic_model = BERTopic.load("TM_Test_Regroup_Camembert_20")


### Répartitions des partis par topic

In [73]:
topics_per_class = topic_model.topics_per_class(
    docs, classes=df["groupe&gvt_affiliation"]
)

In [74]:
fig_topics_per_class = topic_model.visualize_topics_per_class(
    topics_per_class, top_n_topics=15
)
fig_topics_per_class

### Topics par partis

In [82]:
import pandas as pd

df_stat = topics_per_class.copy()
df_stat = df_stat[df_stat["Topic"] != -1]

# normalisation par parti (recommandée)
df_stat["prop"] = (
    df_stat["Frequency"]
    / df_stat.groupby("Class")["Frequency"].transform("sum")
)


In [83]:
df_stat["Topic_name"] = df_stat["Topic"].map(topic_labels)


In [ ]:
df_stat

In [90]:
TOP_N = 13

df_top = (
    df_stat.sort_values(["Class", "prop"], ascending=[True, False])
      .groupby("Class")
      .head(TOP_N)
)


In [91]:
import plotly.graph_objects as go

parties = df_top["Class"].unique()

fig = go.Figure()

# Ajouter une trace par parti (une seule visible au départ)
for i, party in enumerate(parties):
    df_p = df_top[df_top["Class"] == party]

    fig.add_trace(
        go.Bar(
            x=df_p["prop"],
            y=df_p["Topic_name"],
            orientation="h",
            visible=(i == 0),
            name=party
        )
    )

# Boutons du menu déroulant
buttons = []
for i, party in enumerate(parties):
    visibility = [False] * len(parties)
    visibility[i] = True

    buttons.append(
        dict(
            label=party,
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"Top topics – {party}"}
            ]
        )
    )

fig.update_layout(
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=1.15,
        y=1
    )],
    title=f"Top topics – {parties[0]}",
    xaxis_title="Importance relative",
    yaxis_title="Topics",
    height=500,
    margin=dict(l=200)
)

fig.show()


## Étapes 5 : Analyses diachroniques

* `global_tuning` : Tuning général
  * Indique s'il faut calculer la moyenne de la représentation d'un sujet à l'instant *t* avec sa représentation globale.
* `evolution_tuning` : Tuning évolutif
  * Indique s'il faut calculer la moyenne de la représentation d'un sujet à l'instant *t* avec la représentation de ce sujet à l'instant *t-1*.
* `nr_bins`
  * Nombre de compartiments dans lesquels placer les horodatages. Il est inefficace sur le plan informatique d'extraire les sujets à des milliers d'horodatages différents. Il est donc conseillé de maintenir cette valeur en dessous de 20.


In [92]:
topics_over_time = topic_model.topics_over_time(
    docs,
    timestamps=df["dateSeance_day"],
    global_tuning=True,
    evolution_tuning=True,
    nr_bins=100,
)

In [93]:
fig_dynamic_topic = topic_model.visualize_topics_over_time(
    topics_over_time, top_n_topics=15
)
fig_dynamic_topic

### Exportation : 

In [30]:
topics, probs = topic_model.transform(docs)

In [33]:
import numpy as np
print(type(probs))
print(np.array(probs).shape)


<class 'numpy.ndarray'>
(11429,)


In [37]:
print(probs)


[0.74020581 0.         0.         ... 0.         0.77020859 1.        ]


In [35]:
df_probs = pd.DataFrame(probs)

In [38]:
df_topics = pd.DataFrame(
    probs,
    columns=[f"topic_{i}" for i in range(probs.shape[1])]
)

IndexError: tuple index out of range

In [ ]:
k = 2

df_binary = (
    df
    .apply(lambda x: x >= x.nlargest(k).min(), axis=1)
    .astype(int)
)


In [ ]:
theme_names = {
    "topic_0": "territoires",
    "topic_1": "laicité",
    "topic_2": "éducation",
    "topic_3": "répressif",
    "topic_4": "représentation",
    "topic_5": "santé",
    "topic_6": "symbolique-europe",
    "topic_7": "budget",
    "topic_8": "intégration",
    "topic_9": "PE",
    "topic_10": "droits (femmes-minorités)",
    "topic_11": "vie parlementaire",
    "topic_12": "mémoire",
    "topic_13": "antisémitisme",
    
}

df_binary = df_binary.rename(columns=theme_names)

## Tenter version solide en local

In [ ]:
import math
import numpy as np
import os
import traceback
from datasets import Dataset, load_from_disk
from sentence_transformers import SentenceTransformer
import torch
from torch.cuda import is_available as cuda_available
from torch.cuda import synchronize, empty_cache
from gc import collect as gc_collect
import pandas as pd

# ---------- Config ----------
DATASET_PATH = "path/to/your/dataset"   # dossier créé par save_to_disk
OUT_DIR = "embeddings_output"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # adapte
BATCH_SIZE = 64
CHUNK_SIZE_TOKENS = 256   # taille de fenêtre (tokens) pour chunking
OVERLAP_TOKENS = 32
USE_NORMALIZE = True
DEVICE = "cuda" if cuda_available() else "cpu"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- Helpers ----------
def simple_preprocess(texts):
    """Nettoyage basique: supprime None, strip, lowercase"""
    out = []
    for t in texts:
        if t is None: 
            out.append("")   # ou skip
        else:
            out.append(str(t).strip())
    return out

def chunk_text(text, tokenizer, chunk_size=256, overlap=32):
    """Découpe `text` en morceaux de tokens compatibles avec tokenizer. 
       Retourne list[str] de chunks (reconstruit en substrings via tokenizer.decode)."""
    # Tokenize into ids
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        piece = tokenizer.decode(tokens[start:end], skip_special_tokens=True, clean_up_tokenization_spaces=True)
        chunks.append(piece)
        if end == len(tokens):
            break
        start = end - overlap
    return chunks

def texts_to_chunks(texts, tokenizer, chunk_size, overlap):
    """Pour une liste de textes, construit une liste de chunks et mémorise les indexs
       pour ré-agréger."""
    all_chunks = []
    mapping = []  # mapping[i] = (start_index_in_all_chunks, num_chunks_for_text_i)
    idx = 0
    for t in texts:
        chunks = chunk_text(t, tokenizer, chunk_size, overlap)
        all_chunks.extend(chunks)
        mapping.append((idx, len(chunks)))
        idx += len(chunks)
    return all_chunks, mapping

def aggregate_embeddings(mapping, all_embeddings, method="mean"):
    """Recrée une embedding par texte à partir des embeddings de chunks.
       mapping: list of (start, count). all_embeddings: np.array (N_chunks, dim)."""
    outs = []
    for start, count in mapping:
        if count == 0:
            outs.append(np.zeros(all_embeddings.shape[1], dtype=np.float32))
        else:
            chunk_embs = all_embeddings[start:start+count]
            if method == "mean":
                emb = chunk_embs.mean(axis=0)
            elif method == "max":
                emb = chunk_embs.max(axis=0)
            else:
                emb = chunk_embs.mean(axis=0)
            outs.append(emb)
    return np.vstack(outs)

# ---------- Load dataset ----------
ds = load_from_disk(DATASET_PATH)  # ou Dataset.load_from_disk
texts_raw = ds["texts"]   # adapte le nom de la colonne
texts = simple_preprocess(texts_raw)

# ---------- Init modèle ----------
model = SentenceTransformer(MODEL_NAME, device=DEVICE, trust_remote_code=False)
tokenizer = model.tokenizer  # tokenizer huggingface du modèle
# Optionnel : fixer max_seq_length si tu veux tronquer
model.max_seq_length = min(model.max_seq_length, 512)  # adapte

# ---------- Chunking si nécessaire ----------
# Si tu as beaucoup de textes courts, tu peux ignorer le chunking et encoder directement
need_chunking = any(len(tokenizer.encode(t, add_special_tokens=False)) > model.max_seq_length for t in texts)
if need_chunking:
    print("Chunking texts because some exceed max_seq_length...")
    all_chunks, mapping = texts_to_chunks(texts, tokenizer, CHUNK_SIZE_TOKENS, OVERLAP_TOKENS)
    to_encode = all_chunks
else:
    to_encode = texts
    mapping = [(i,1) for i in range(len(texts))]

# ---------- Encode par batch ----------
def batched_encode(model, items, batch_size=BATCH_SIZE, normalize=USE_NORMALIZE, device=DEVICE):
    embeddings = []
    for i in range(0, len(items), batch_size):
        batch = items[i:i+batch_size]
        emb = model.encode(batch, batch_size=len(batch), device=device, normalize_embeddings=normalize, convert_to_numpy=True, show_progress_bar=False)
        embeddings.append(emb)
    return np.vstack(embeddings)

try:
    all_embeddings = batched_encode(model, to_encode, batch_size=BATCH_SIZE, normalize=USE_NORMALIZE, device=DEVICE)
    # Si chunking, agréger
    if need_chunking:
        final_embeddings = aggregate_embeddings(mapping, all_embeddings, method="mean")
    else:
        final_embeddings = all_embeddings  # shape (N_texts, dim)

    # Sauvegarde efficace: numpy memmap + dataset parquet
    emb_path = os.path.join(OUT_DIR, "embeddings.npy")
    np.save(emb_path, final_embeddings)  # petite dataset ok
    # Si très grand: np.memmap + écriture progressive serait plus adapté

    # Ajout au dataset (convert to list)
    ds2 = ds.add_column("embedding", [row.tolist() for row in final_embeddings])
    ds2.save_to_disk(os.path.join(OUT_DIR, "dataset_with_embeddings"))
    print("Saved embeddings and dataset successfully.")
except Exception as e:
    print("Error during encoding:")
    traceback.print_exc()
finally:
    # cleanup
    del model
    if DEVICE.startswith("cuda"):
        empty_cache()
        try:
            synchronize()
        except Exception:
            pass
    gc_collect()
